# Mamba4Cast Baseline vs SC-Mamba Data Provider

This unified notebook environment strictly runs the original unaltered `automl/Mamba4Cast` model but feeds it data natively constructed from the `ngngsonan/SC-Mamba` pipeline (NASDAQ-100 sequences). 
It logs deterministic baseline metrics (MASE, MAE) and dumps raw output arrays for later comparative visual analysis without re-execution.

## 1. Environment Resolution (Colab T4)
Uninstall default Colab PyTorch libraries and install PyTorch 2.4.0 (cu121) perfectly matched to `causal_conv1d` and `mamba_ssm` compiled wheels.

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.39.3 packaging triton

# Downloading compiled causal CUDNN wheels to prevent 45+ min ninja compile times
!wget -qO causal_conv1d.whl "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
!wget -qO mamba_ssm.whl "https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

!pip install causal_conv1d.whl mamba_ssm.whl
!pip install yfinance gluonts tqdm utilsforecast

## 2. Clone Repositories & Download Weights
Clone both SC-Mamba (for the custom data API) and Mamba4Cast (for the pure model weights).

In [ ]:
!git clone https://github.com/ngngsonan/SC-Mamba.git
!git clone https://github.com/automl/Mamba4Cast.git

!mkdir -p Mamba4Cast/models
!wget -O Mamba4Cast/models/mamba4cast_2l_1024_conv_i5e5.pth "https://www.dropbox.com/scl/fi/s7l0razs4xdctcbxkik5t/mamba4cast_2l_1024_conv_i5e5.pth?rlkey=khd27b5tw4ipp06pt13vgl1l5&st=a3bfak3v&dl=1"

## 3. Generate Target Dataset
Trigger the `store_real_datasets.py` pipeline native to SC-Mamba to retrieve the NASDAQ `.pkl` caches.

In [ ]:
# Generates inside SC-Mamba/SC-Mamba-Code/data/real_val_datasets/nasdaq_nopad_512.pkl
!cd SC-Mamba/SC-Mamba-Code && python data/scripts/store_real_datasets.py

## 4. Cross-Repository Inference Generator
We write a runner script `run_hybrid_baseline.py` dynamically tracking module imports spanning both repos simultaneously.

In [ ]:
%%writefile run_hybrid_baseline.py
import os
import sys
import torch
import pandas as pd
from tqdm import tqdm
from utilsforecast.losses import mase, mae, smape, rmse

# 1. Force python path routing across both repositories
sys.path.append(os.path.abspath('Mamba4Cast/src_torch'))
sys.path.append(os.path.abspath('SC-Mamba/SC-Mamba-Code'))
sys.path.append(os.path.abspath('SC-Mamba/SC-Mamba-Code/data'))

from training.models import SSMModelMulti  # From Original Mamba4Cast
from data_provider.data_factory import data_provider  # From SC-Mamba

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Evaluating strict Mamba4Cast Baseline on {device}")

ssm_config = {
    "bidirectional": False,
    "enc_conv": True,
    "init_dil_conv": True,
    "enc_conv_kernel": 5,
    "init_conv_kernel": 5,
    "init_conv_max_dilation": 3,
    "global_residual": False,
    "in_proj_norm": False,
    "initial_gelu_flag": True,
    "linear_seq": 15,
    "mamba2": True,
    "norm": True,
    "norm_type": "layernorm",
    "num_encoder_layers": 2,
    "d_state": 128,
    "residual": False,
    "token_embed_len": 1024,
}

model_path = 'Mamba4Cast/models/mamba4cast_2l_1024_conv_i5e5.pth'
subday = True

def adapt_state_dict_keys(old_state_dict):
    new_state_dict = {}
    for key in old_state_dict.keys():
        if "linear_layer" in key:
            layer_idx = key.split('.')[1]
            new_key = key.replace(f"linear_layer", f"stage_2_layer.0")
            new_state_dict[new_key] = old_state_dict[key]
        else:
            new_state_dict[key] = old_state_dict[key]
    return new_state_dict

def scale_data(output, scaler):
    if scaler == 'min_max':
        output = (output['result'] * (output['scale'][0].squeeze(-1) - output['scale'][1].squeeze(-1))) + output['scale'][1].squeeze(-1)
    return output

model = SSMModelMulti(scaler='min_max', sub_day=subday, **ssm_config).to(device)
new_state_dict = adapt_state_dict_keys(torch.load(model_path, map_location=device)['model_state_dict'])
model.load_state_dict(new_state_dict, strict=False)
model.eval()

args = {
    'data': 'nasdaq',
    'root_path': 'SC-Mamba/SC-Mamba-Code/data/real_val_datasets/',
    'data_path': 'nasdaq_nopad_512.pkl',
    'seq_len': 512,
    'label_len': 0,
    'pred_len': 30,
    'embed': 'timeF',
    'freq': 'B',
    'num_workers': 2
}

dataset, dataloader = data_provider(args, 'test', subday=subday)

batch_pred_dfs = []
with torch.no_grad():
    for i, batch in tqdm(enumerate(dataloader), total=len(dataset)):
        cl = min(512, batch['x'].shape[1])
        ids = batch["id"]
        batch_x = batch["x"][:, -cl:, :].float().to(device)
        batch_y = batch["y"][:, :30, :].squeeze().float().to(device)
        batch_x_mark = batch["ts_x"][:, -cl:, :].float().to(device)
        batch_y_mark = batch["ts_y"][:, :30, :].float().to(device)

        x = {'ts': batch_x_mark, 'history': batch_x.reshape(1, batch_x.size(1)), 'target_dates': batch_y_mark, 'task': torch.zeros(1, 30).int().to(device)}

        output = model(x, prediction_length=30)
        scaled_output = scale_data(output, 'min_max').detach().cpu().squeeze()

        batch_pred_dfs.append(pd.DataFrame({
            'id': ids.repeat_interleave(30).numpy(),
            'pred': scaled_output.flatten().numpy(),
            'target': batch_y.cpu().flatten().numpy()
        }))

pred_df = pd.concat(batch_pred_dfs)

# Output Results
mae_loss = mae(pred_df, ['pred'], 'id', 'target')
rmse_loss = rmse(pred_df, ['pred'], 'id', 'target')
smape_loss = smape(pred_df, ['pred'], 'id', 'target')
mase_loss = mase(pred_df, ['pred'], 'id', 'target', seasonality=1) # Approximate seasonality 1 for daily

print("\n=== Mamba4Cast Baseline Zero-Shot Performance on NASDAQ ===")
print(f"MAE:   {mae_loss['pred'].mean():.4f}")
print(f"RMSE:  {rmse_loss['pred'].mean():.4f}")
print(f"SMAPE: {smape_loss['pred'].mean():.4f}")
print(f"MASE:  {mase_loss['pred'].mean():.4f}")

os.makedirs("SC-Mamba/SC-Mamba-Code/benchmark", exist_ok=True)
pred_df.to_csv("SC-Mamba/SC-Mamba-Code/benchmark/baseline_predictions.csv", index=False)
print("Saved raw granular predictive insights to SC-Mamba/SC-Mamba-Code/benchmark/baseline_predictions.csv")


## 5. Execute Validation and Save Output Metrics


In [ ]:
!python run_hybrid_baseline.py